# Python Foundations: Built-ins, Methods, Formatting, and Everyday Engineering Patterns

Maps to `design3.md` Phase 1.

This notebook is the first Python foundation notebook. It is meant to make you fluent with the objects and operations you will use every day in Python: `str`, `list`, `tuple`, `set`, `frozenset`, `dict`, plus the small set of standard-library tools that show up constantly in data and platform work.

The target is not memorizing trivia. The target is being able to read and write ordinary Python without friction, choose the right container deliberately, and explain your choices in an interview.


## How To Use This Notebook

Work through the notebook in order.

For each major section:
- read the explanation first
- run the example code
- change one thing in the example and observe what changes
- answer the interview-style questions in your own words before moving on

What "good Python foundation" means here:
- you know the core built-ins and their common methods
- you know how Python formatting works well enough to produce clean logs, reports, and debug output
- you can choose between `list`, `tuple`, `set`, and `dict` quickly and defend the choice
- you can solve common ingestion, normalization, lookup, aggregation, and reporting problems using only the standard library


## Quick Container Selection Guide

| Need | Best default | Why |
| --- | --- | --- |
| Ordered sequence you will mutate | `list` | Append, update, sort, slice |
| Fixed-size record or hashable composite key | `tuple` | Immutable and can often be used as a dict key |
| Fast membership / dedupe / set algebra | `set` | Uniqueness plus efficient membership testing |
| Immutable set that itself must be hashable | `frozenset` | Useful as a dict key or set element |
| Key-value lookup | `dict` | The normal mapping type in Python |
| Queue / sliding buffer | `collections.deque` | Better than `list.pop(0)` |
| Counting | `collections.Counter` | Specialized counting API |
| Grouping into buckets | `collections.defaultdict` | Removes repeated missing-key boilerplate |

Interview version of the same question:
- Why is a `set` usually better than a `list` for repeated membership checks?
- Why is a `tuple` acceptable as a dictionary key but a `list` is not?
- Why is `deque` better than `list` for queue workloads?


## 1. Strings

`str` is one of the most important Python types. Strings are immutable Unicode text. That gives you two important rules:
- every apparent "modification" produces a new string object
- many string operations are safe to chain because they do not mutate the original value

Why strings matter in industry:
- normalize raw vendor fields and user input
- parse delimited text
- build filenames, SQL fragments, log lines, and reports
- format numeric and datetime values for human output

You should know these string ideas cold:
- trimming and cleanup: `strip`, `lstrip`, `rstrip`
- splitting and partitioning: `split`, `rsplit`, `splitlines`, `partition`, `rpartition`
- search and replacement: `replace`, `find`, `index`, `count`
- comparisons and normalization: `lower`, `upper`, `casefold`, `startswith`, `endswith`
- joining: `sep.join(parts)`
- formatting: f-strings, `str.format`, and `format(value, spec)`
- prefix/suffix cleanup: `removeprefix`, `removesuffix`


In [ ]:
from datetime import datetime, timezone

raw_line = " eurusd | ecn | trade | 1.08450 | 2026-04-04T08:15:00+00:00 "
parts = [part.strip() for part in raw_line.split("|")]

symbol = parts[0].upper()
venue = parts[1].casefold().upper()
event_type = parts[2].strip().upper()
price = float(parts[3])
ts = datetime.fromisoformat(parts[4]).astimezone(timezone.utc)

normalized = {
    "symbol": symbol,
    "venue": venue,
    "event_type": event_type,
    "price": price,
    "ts": ts,
}

normalized


### String Methods You Should Actually Use

| Method | What it does | Typical use |
| --- | --- | --- |
| `strip()` / `lstrip()` / `rstrip()` | Remove surrounding whitespace or characters | Clean CSV fields or raw text input |
| `split(sep)` / `rsplit(sep)` | Split into pieces | Parse delimited input |
| `partition(sep)` / `rpartition(sep)` | Split into exactly 3 parts | Safer when you want "before, separator, after" |
| `splitlines()` | Split on line boundaries | Multi-line file or log parsing |
| `replace(old, new)` | Replace text | Cleanup or normalization |
| `find(sub)` | Return index or `-1` | Safe search without exception |
| `index(sub)` | Return index or raise `ValueError` | Use when missing should be treated as an error |
| `count(sub)` | Count occurrences | Quality checks and light parsing |
| `startswith(prefix)` / `endswith(suffix)` | Boundary check | File extensions, prefixes, routing |
| `lower()` / `upper()` / `casefold()` | Case normalization | `casefold()` for stronger case-insensitive comparison |
| `join(iterable)` | Concatenate with separator | Build CSV lines, paths, messages |
| `removeprefix(x)` / `removesuffix(x)` | Remove expected boundary text | Symbol or filename cleanup |
| `isalpha()` / `isdigit()` / `isalnum()` | Character-class checks | Quick validation |

Two high-value distinctions:
- `split` vs `partition`: use `partition` when you want a stable three-part result even if the separator appears once or not at all
- `find` vs `index`: `find` returns `-1` if missing, `index` raises an exception


In [ ]:
filename = "prices_2026-04-04.csv"
name, dot, extension = filename.partition(".")

search_demo = {
    "find_trade": "trade|quote|error".find("quote"),
    "count_pipe": raw_line.count("|"),
    "filename_base": name,
    "filename_extension": extension,
    "removed_suffix": filename.removesuffix(".csv"),
}

search_demo


### Formatting: f-strings, `str.format`, and `format()`

You specifically called this out, and you are right: formatting deserves explicit treatment.

All three are related:
- **f-strings** are usually the best default in modern Python because they are readable and direct
- **`str.format(...)`** is older but still common in existing codebases and useful when the format string is built separately
- **`format(value, spec)`** is the underlying formatting protocol for a single value

The important thing is the **format specification mini-language**.

Common specs you should know:
- `:.2f` -> fixed-point with 2 decimals
- `:,.2f` -> thousands separator plus 2 decimals
- `:>12` -> right-align in width 12
- `:<12` -> left-align in width 12
- `:^12` -> center-align in width 12
- `:08d` -> zero-pad integer to width 8
- `:.2%` -> percentage with 2 decimals

Interview question you should be able to answer:
- What is the relationship between f-strings and the format mini-language?
  The short answer: f-strings use the same formatting protocol, so `f"{x:,.2f}"` and `format(x, ',.2f')` are using the same format spec.


In [ ]:
price = 145.123456
notional = 12_500_000.5
position = 42
report_ts = datetime(2026, 4, 4, 16, 30, tzinfo=timezone.utc)

formatted = {
    "f_string": f"price={price:.4f} notional={notional:,.2f} position={position:04d}",
    "str_format": "price={:.4f} notional={:,.2f} position={:04d}".format(price, notional, position),
    "format_builtin": format(notional, ",.2f"),
    "alignment": f"|{'EURUSD':<10}|{'BUY':^8}|{price:>10.4f}|",
    "percent": f"fill_rate={0.97325:.2%}",
    "datetime": f"ts={report_ts:%Y-%m-%d %H:%M:%S %Z}",
}

formatted


### String Interview Questions

Answer these without searching:
- Why is `join` called on the separator string rather than on the list?
- When would you use `partition` instead of `split`?
- What is the difference between `find` and `index`?
- Why is `casefold()` stronger than `lower()` for case-insensitive comparisons?
- When would you use `f"{value:,.2f}"` instead of converting the number manually?
- How would you format a timestamp, an aligned table row, and a comma-separated money value?


## 2. Lists and Tuples

Lists are ordered, mutable sequences. Tuples are ordered, immutable sequences.

Use a `list` when:
- order matters
- duplicates are allowed
- you expect to append, update, sort, or slice

Use a `tuple` when:
- the record shape is fixed
- mutation would be misleading or dangerous
- you want a composite key that can be hashed

Industry uses:
- lists for batches of parsed rows, ordered event streams, and report rows
- tuples for keys like `(symbol, venue)` or `(date, region)`


In [ ]:
prices = [1.0845, 1.0847, 1.0846]
prices.append(1.0848)
prices.extend([1.0849, 1.0850])

record = ("EURUSD", "ECN", 1.0845)
symbol, venue, px = record

list_tuple_demo = {
    "prices": prices,
    "top_two_desc": sorted(prices, reverse=True)[:2],
    "last_price": prices[-1],
    "record": record,
    "unpacked": (symbol, venue, px),
}

list_tuple_demo


### List Methods You Should Know Cold

| Method | What it does | Typical use |
| --- | --- | --- |
| `append(x)` | Add one item to the end | Build batches incrementally |
| `extend(iterable)` | Add many items | Merge batches or append another sequence |
| `insert(i, x)` | Insert at position | Rare in hot paths; shifts later items |
| `remove(x)` | Remove first matching value | Cleanup by value |
| `pop()` / `pop(i)` | Remove and return item | Stack-like behavior or indexed removal |
| `clear()` | Remove all items | Reset a batch |
| `index(x)` | Find first index | Rare; costs a scan |
| `count(x)` | Count matches | Quick summary checks |
| `sort(...)` | In-place sort | Efficient when you want to mutate the list |
| `reverse()` | In-place reverse | Reverse order without creating a new list |
| `copy()` | Shallow copy | Separate top-level list object |

High-value ideas:
- slicing creates a new list: `values[:3]`
- list comprehensions are often clearer than `map` for straightforward transformations
- lists are good stacks: `append` + `pop()`
- lists are poor queues when you keep doing `pop(0)` because the remaining items shift left


In [ ]:
rows = [
    {"symbol": "EURUSD", "price": 1.0845, "ts": 3},
    {"symbol": "USDJPY", "price": 145.12, "ts": 1},
    {"symbol": "GBPUSD", "price": 1.2641, "ts": 2},
]

sorted_rows = sorted(rows, key=lambda row: row["ts"])
price_only = [row["price"] for row in sorted_rows]
copy_of_prices = prices.copy()
copy_of_prices.pop()

list_methods_demo = {
    "sorted_rows": sorted_rows,
    "price_only": price_only,
    "original_prices": prices,
    "copy_after_pop": copy_of_prices,
}

list_methods_demo


### Tuple Features That Matter

Tuples are simpler, but three tuple patterns matter a lot:
- **packing**: `pair = symbol, venue`
- **unpacking**: `symbol, venue = pair`
- **composite keys**: `lookup[(symbol, venue)] = metadata`

Interview questions:
- Why might you choose a tuple over a list for a fixed-shape record?
- Why is `(symbol, venue)` a good dictionary key but `[symbol, venue]` is not?
- What does "shallow copy" of a list mean if the list contains dictionaries?


## 3. Sets and Frozensets

Sets are about uniqueness and membership. They are one of the best tools for turning an O(n) repeated membership problem into something far cheaper in practice.

Use a `set` when:
- duplicates are meaningless or should be removed
- membership testing is frequent
- you need union, intersection, or difference

Use a `frozenset` when:
- you need set semantics but the object itself must be immutable and hashable


In [ ]:
expected_symbols = {"EURUSD", "USDJPY", "GBPUSD", "AUDUSD"}
observed_symbols = ["EURUSD", "USDJPY", "EURUSD", "AUDUSD", "EURUSD"]
observed_set = set(observed_symbols)

set_demo = {
    "observed_unique": observed_set,
    "missing": expected_symbols - observed_set,
    "extra": observed_set - expected_symbols,
    "common": expected_symbols & observed_set,
    "union": expected_symbols | observed_set,
    "frozenset_key": frozenset({"LONDON", "NY"}),
}

set_demo


### Set Methods You Should Know Cold

| Method | What it does | Typical use |
| --- | --- | --- |
| `add(x)` | Add one element | Maintain seen IDs or symbols |
| `update(iterable)` | Add many elements | Bulk ingest of identifiers |
| `remove(x)` | Remove or raise `KeyError` | Use when missing is exceptional |
| `discard(x)` | Remove if present | Safer when missing is normal |
| `pop()` | Remove arbitrary element | Rare in business logic |
| `clear()` | Empty the set | Reset dedupe state |
| `union(...)` / `|` | Combine elements | Coverage comparison |
| `intersection(...)` / `&` | Shared elements | Common symbol universe |
| `difference(...)` / `-` | In left, not right | Missing records |
| `symmetric_difference(...)` / `^` | In exactly one side | Reconciliation mismatch |
| `issubset(...)` / `issuperset(...)` | Set relation | Validation or coverage checks |

Interview questions:
- Why is a set usually better than a list for repeated membership checks?
- What is the difference between `remove` and `discard`?
- What business question is naturally expressed as a set difference?


## 4. Dictionaries

`dict` is the default Python mapping type. Modern dictionaries preserve insertion order, which makes them useful both as lookup tables and as stable containers for structured data.

Why dictionaries matter:
- almost every non-trivial Python system uses them constantly
- configuration, parsed JSON, routing tables, aggregations, counters, caches, and lookup tables are all naturally dictionary-shaped

Important dictionary ideas:
- keys must be hashable
- values can be anything
- `get` is safer than direct indexing when missing keys are normal
- `items()` is the normal way to iterate key-value pairs
- dictionary comprehensions are useful for building lookup tables cleanly


In [ ]:
quotes = [
    {"symbol": "EURUSD", "bid": 1.0845, "ask": 1.0847},
    {"symbol": "USDJPY", "bid": 145.12, "ask": 145.14},
    {"symbol": "GBPUSD", "bid": 1.2641, "ask": 1.2644},
]

quote_by_symbol = {row["symbol"]: row for row in quotes}
spread_by_symbol = {
    symbol: round(row["ask"] - row["bid"], 5)
    for symbol, row in quote_by_symbol.items()
}

config_base = {"timeout": 5, "retries": 3}
config_override = {"timeout": 10}
merged_config = config_base | config_override

quote_by_symbol, spread_by_symbol, merged_config


### Dictionary Methods You Should Know Cold

| Method | What it does | Typical use |
| --- | --- | --- |
| `d[key]` | Direct lookup or `KeyError` | Use when key must exist |
| `get(key, default)` | Safe lookup with fallback | Missing keys are normal |
| `setdefault(key, default)` | Get existing or insert default | Occasional grouping setup |
| `update(other)` | Merge in another mapping | Config overlays |
| `pop(key, default)` | Remove and return value | Consume state or cleanup |
| `popitem()` | Remove last inserted item | Stack-like mapping use, rare |
| `keys()` / `values()` / `items()` | Dictionary views | Iteration and summaries |
| `copy()` | Shallow copy | Separate top-level mapping |
| `fromkeys(iterable, value)` | Build keys with same value | Sometimes useful for initialization |

Two subtle points worth knowing:
- `1`, `1.0`, and `True` compare equal as keys, so they refer to the same dictionary slot
- `dict.copy()` is a shallow copy; nested lists or dicts are still shared


In [ ]:
records = [
    {"symbol": "EURUSD", "venue": "ECN"},
    {"symbol": "EURUSD", "venue": "BANK"},
    {"symbol": "USDJPY", "venue": "ECN"},
]

by_symbol = {}
for record in records:
    by_symbol.setdefault(record["symbol"], []).append(record["venue"])

counts = {}
for record in records:
    symbol = record["symbol"]
    counts[symbol] = counts.get(symbol, 0) + 1

nested = {"symbols": ["EURUSD", "USDJPY"]}
shallow_copy = nested.copy()
shallow_copy["symbols"].append("GBPUSD")

{
    "grouped_with_setdefault": by_symbol,
    "counts_with_get": counts,
    "shallow_copy_effect": nested,
}


### Dictionary Interview Questions

Answer these without searching:
- When should you use `d[key]` instead of `d.get(key)`?
- What problem does `setdefault` solve, and when is `defaultdict` cleaner?
- Why must dictionary keys be hashable?
- What does it mean that dictionaries preserve insertion order?
- What is a shallow copy of a dictionary?


## 5. Built-ins and Iteration Helpers You Will Use Constantly

These show up in good Python all the time:
- `enumerate(iterable)` for index + value
- `zip(a, b, ...)` for parallel iteration
- `sorted(iterable, key=...)` for non-mutating sort
- `min(..., key=...)` / `max(..., key=...)` for selection by criteria
- `any(...)` / `all(...)` for boolean aggregation
- unpacking: `head, *middle, tail`


In [ ]:
trades = [
    {"symbol": "EURUSD", "price": 1.0845},
    {"symbol": "USDJPY", "price": 145.12},
    {"symbol": "GBPUSD", "price": 1.2641},
]

symbols = [trade["symbol"] for trade in trades]
prices_only = [trade["price"] for trade in trades]

helper_demo = {
    "enumerate": list(enumerate(symbols, start=1)),
    "zip": list(zip(symbols, prices_only)),
    "sorted_by_length": sorted(symbols, key=len),
    "max_price_trade": max(trades, key=lambda row: row["price"]),
    "all_positive": all(price > 0 for price in prices_only),
    "any_over_100": any(price > 100 for price in prices_only),
}

helper_demo


## 6. Standard Library Workhorses

You should know these before reaching for a bigger dependency.

### `collections.Counter`
Use when the real job is counting.

### `collections.defaultdict`
Use when missing keys should create buckets automatically.

### `collections.deque`
Use for queues and sliding windows.

### `pathlib.Path`
Use as the normal filesystem path abstraction.

### `csv` and `json`
Use heavily for fixtures, lightweight ingestion, exports, and local tooling.

### `itertools`
Use for iterator-based transformations and combinatorics.


In [ ]:
from collections import Counter, defaultdict, deque
from pathlib import Path
import csv
import io
import itertools
import json

rows = [
    {"symbol": "EURUSD", "venue": "ECN", "event_type": "TRADE", "price": 1.0845},
    {"symbol": "EURUSD", "venue": "BANK", "event_type": "TRADE", "price": 1.0846},
    {"symbol": "USDJPY", "venue": "ECN", "event_type": "QUOTE", "price": 145.12},
]

event_counts = Counter(row["event_type"] for row in rows)
venues_by_symbol = defaultdict(set)
for row in rows:
    venues_by_symbol[row["symbol"]].add(row["venue"])

queue = deque(["task-1", "task-2", "task-3"])
first_task = queue.popleft()

csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=["symbol", "venue", "event_type", "price"])
writer.writeheader()
writer.writerows(rows)

json_text = json.dumps(rows, indent=2)
price_values = [row["price"] for row in rows]
pairs = list(zip(price_values, price_values[1:]))
example_path = Path("data") / "sample" / "quotes.json"

{
    "event_counts": event_counts,
    "venues_by_symbol": {k: sorted(v) for k, v in venues_by_symbol.items()},
    "first_task": first_task,
    "csv_preview": csv_buffer.getvalue().splitlines()[:3],
    "json_preview": json_text.splitlines()[:6],
    "pairwise_prices": pairs,
    "example_path": str(example_path),
}


### Standard Library Interview Questions

- When would you use `Counter` instead of a normal dictionary?
- When is `defaultdict(list)` cleaner than `setdefault`?
- Why is `deque` better than `list` for queue-like workloads?
- When is `Path` better than string-based path manipulation?
- When can `csv` and `json` solve the job without pulling in a dataframe library?


## 7. Comprehensions: List, Dict, Set, and Generator Expressions

Comprehensions are concise, readable, and faster than equivalent `for` loops in CPython. They are in nearly every Python codebase in the industry.

**Four forms:**

| Form | Syntax | Produces | Lazy? |
| --- | --- | --- | --- |
| List comprehension | `[expr for x in it if cond]` | `list` | No |
| Dict comprehension | `{k: v for k, v in it if cond}` | `dict` | No |
| Set comprehension | `{expr for x in it}` | `set` | No |
| Generator expression | `(expr for x in it if cond)` | Generator | **Yes** |

**Key rules:**
- Comprehensions evaluate eagerly (except generators) — the whole result is in memory
- Generator expressions are lazy — values are produced on demand, one at a time
- Use a generator expression when you only need the aggregate (e.g., `sum(x for x in ...)`)
- Use a regular `for` loop when the body has multiple side effects or multi-step logic
- Avoid deeply nested comprehensions — they hurt readability

**Nested comprehension pattern (flattening):**
```python
[cell for row in matrix for cell in row]  # reads as nested for-loops
```

**Industry uses:**
- Normalize a batch of raw dicts in one expression
- Build `symbol → price` lookup tables from record lists
- Deduplicate identifiers with a set comprehension
- Lazy-evaluate large dataset aggregations with generator expressions

**Interview questions:**
- What is the difference between a list comprehension and a generator expression?
- When would a generator expression be better than a list comprehension?
- What is the memory implication of `[x for x in range(10_000_000)]` vs `(x for x in range(10_000_000))`?
- How do you read a nested comprehension like `[x for row in matrix for x in row]`?

In [ ]:
from decimal import Decimal

raw_trades = [
    {"symbol": "eurusd", "price": "1.0845", "volume": "1000", "side": "BUY"},
    {"symbol": " usdjpy ", "price": "145.12", "volume": "500", "side": "sell"},
    {"symbol": "EURUSD", "price": "1.0847", "volume": "2000", "side": "BUY"},
    {"symbol": "gbpusd", "price": "1.2641", "volume": "750", "side": "BUY"},
]

# List comprehension — most common form
symbols = [row["symbol"].strip().upper() for row in raw_trades]

# List comprehension with condition (filter)
buy_rows = [row for row in raw_trades if row["side"].upper() == "BUY"]

# Dict comprehension — build lookup table
latest_price = {
    row["symbol"].strip().upper(): Decimal(row["price"])
    for row in raw_trades
}

# Set comprehension — unique values, deduplicated automatically
unique_symbols = {row["symbol"].strip().upper() for row in raw_trades}

# Generator expression — lazy; does NOT build a list in memory
# Use for large datasets or when you only need the aggregate
total_notional = sum(
    Decimal(row["price"]) * int(row["volume"])
    for row in raw_trades
)

# Conditional expression inside comprehension (ternary-style)
normalized_sides = [
    "BUY" if row["side"].upper() == "BUY" else "SELL"
    for row in raw_trades
]

# Nested comprehension — flatten a 2D structure
batch_of_batches = [["EURUSD", "USDJPY"], ["GBPUSD"], ["AUDUSD", "NZDUSD"]]
all_symbols = [sym for batch in batch_of_batches for sym in batch]

# Dict comprehension from two parallel lists (zip)
col_names = ["symbol", "bid", "ask"]
col_values = ["EURUSD", 1.0845, 1.0847]
row_dict = {k: v for k, v in zip(col_names, col_values)}

# When NOT to use comprehensions: multi-step logic is clearer as a loop
# One transformation = comprehension; multiple transformations = explicit loop

{
    "symbols": symbols,
    "buy_count": len(buy_rows),
    "latest_price": {k: str(v) for k, v in latest_price.items()},
    "unique_symbols": unique_symbols,
    "total_notional": str(total_notional),
    "normalized_sides": normalized_sides,
    "flattened": all_symbols,
    "row_from_zip": row_dict,
}

## 8. Decorators and `functools`

Decorators are one of the most used Python patterns in production. They let you add behavior to a function without modifying it directly — separation of cross-cutting concerns.

**The decorator protocol:**
```
@decorator
def my_func(): ...
# is exactly equivalent to:
my_func = decorator(my_func)
```

**Always use `@functools.wraps`** inside decorators — it preserves the wrapped function's `__name__`, `__doc__`, and `__module__`. Without it, debugging and introspection break.

**Decorator factory** (decorator that takes arguments):
```python
@retry(max_attempts=3)  # retry(...) returns the actual decorator
def fetch(): ...
```

**Stacking decorators**: applied bottom-up, but executed top-down at call time.

**`functools` tools you must know:**

| Tool | What it does | When to use |
| --- | --- | --- |
| `@functools.wraps(f)` | Copy metadata from wrapped function | Always — inside every decorator |
| `@functools.lru_cache(maxsize=N)` | Memoize return values keyed by args | Pure functions with repeated expensive calls |
| `functools.partial(f, **kwargs)` | Freeze some arguments, create specialized callable | Adapters and configuration-driven factories |
| `functools.reduce(f, iterable)` | Left-fold | Rarely needed; prefer loops or `sum` |

**Industry patterns:**
- `@timing` — profiling in local dev and load tests
- `@retry(max_attempts=3)` — resilient API/DB calls
- `@lru_cache` — reference data lookups (instrument metadata, calendar data)
- `functools.partial` — creating specialized pipeline stages

**Interview questions:**
- What does `@functools.wraps` do and why does it matter?
- What is the difference between a decorator and a decorator factory?
- When would you use `lru_cache` and what are its risks?
- What does `functools.partial` return and when is it useful?

In [ ]:
import functools
import time


# --- The decorator protocol: a callable that wraps a function ---
def timing(func):
    @functools.wraps(func)   # preserve __name__, __doc__, __module__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timing] {func.__name__} took {elapsed*1000:.2f}ms")
        return result
    return wrapper


# --- Decorator factory (decorator that takes arguments) ---
def retry(max_attempts: int = 3, delay: float = 0.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
            return None
        return wrapper
    return decorator


# --- Stacking decorators (applied bottom-up, executed top-down) ---
@timing
@retry(max_attempts=2)
def fetch_quote(symbol: str) -> dict:
    return {"symbol": symbol, "price": 1.0845}


# --- functools.lru_cache: memoize pure function results ---
@functools.lru_cache(maxsize=256)
def get_instrument_category(symbol: str) -> str:
    # simulate an expensive lookup
    if len(symbol) == 6 and symbol.isalpha():
        return "FX_SPOT"
    elif symbol.startswith("CL"):
        return "ENERGY_FUTURE"
    return "UNKNOWN"


# --- functools.partial: freeze arguments of a function ---
def apply_fee(price: float, fee_bps: float) -> float:
    return price * (1 + fee_bps / 10_000)

retail_fee = functools.partial(apply_fee, fee_bps=15.0)
institutional_fee = functools.partial(apply_fee, fee_bps=2.5)

quote = fetch_quote("EURUSD")
cat = get_instrument_category("EURUSD")
_ = get_instrument_category("EURUSD")   # cached — no re-computation

{
    "quote": quote,
    "instrument_category": cat,
    "cache_info": str(get_instrument_category.cache_info()),
    "retail_price": round(retail_fee(1.0845), 6),
    "institutional_price": round(institutional_fee(1.0845), 6),
    "original_name_preserved": fetch_quote.__name__,
}


## 9. Context Managers: `with`, `__enter__`/`__exit__`, and `contextlib`

A context manager guarantees setup and teardown — even if an exception is raised. This is how Python handles resources safely without `try/finally` noise everywhere.

**How it works:**
- `with expr as x:` calls `expr.__enter__()` → assigns result to `x`
- On exit (normal or exception), `__exit__(exc_type, exc_val, exc_tb)` is called
- If `__exit__` returns `True`, the exception is suppressed; returning `False` lets it propagate

**Two ways to build context managers:**
1. **Class-based**: implement `__enter__` and `__exit__` — use when you need full control
2. **`@contextlib.contextmanager`**: generator-based — use for simple cases; `yield` is where the `with` body runs

**`contextlib` tools worth knowing:**

| Tool | Purpose |
| --- | --- |
| `@contextlib.contextmanager` | Decorator for generator-based context managers |
| `contextlib.suppress(*excs)` | Silently ignore listed exceptions |
| `contextlib.nullcontext()` | Placeholder no-op (useful for optional context managers) |
| `contextlib.ExitStack` | Manage a dynamic number of context managers |

**Industry uses:**
- Database connections and transaction management
- Acquiring and releasing locks
- Timing/benchmarking blocks
- Temporary files and directories
- Mocking in tests (`unittest.mock.patch` is a context manager)

**Interview questions:**
- What happens to `__exit__` if an exception is raised inside the `with` block?
- What is the difference between `contextlib.contextmanager` and a class-based context manager?
- When would you use `contextlib.suppress`?
- Why is `with open(...)` better than manually calling `file.close()`?


In [ ]:
import contextlib
import time
from pathlib import Path
import tempfile


# --- Class-based context manager ---
class Timer:
    """Measure elapsed time of any block."""
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        return False  # do NOT suppress exceptions — let them propagate


# --- contextlib.contextmanager: simpler generator-based approach ---
@contextlib.contextmanager
def managed_connection(dsn: str):
    """Simulated DB connection with guaranteed cleanup."""
    conn = {"dsn": dsn, "open": True}
    try:
        yield conn          # control passes to `with` body here
    except Exception:
        # opportunity to rollback, log, etc.
        raise
    finally:
        conn["open"] = False  # runs even if exception is raised


# --- contextlib.suppress: cleanly ignore specific exceptions ---
with contextlib.suppress(FileNotFoundError):
    Path("definitely_does_not_exist.txt").unlink()


# --- Nested context managers (common with files + timers) ---
with Timer() as t:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
        for i in range(1000):
            f.write(f"EURUSD,{1.0845 + i * 0.0001:.4f},{i * 100}\n")
        tmp_path = f.name

Path(tmp_path).unlink()

# --- Reentrant context managers with managed_connection ---
with managed_connection("postgres://localhost/trading") as conn:
    was_open = conn["open"]

{
    "write_timer_ms": round(t.elapsed * 1000, 2),
    "connection_was_open_inside": was_open,
    "connection_after_exit": conn["open"],
    "suppress_worked": "FileNotFoundError silently suppressed",
}


## 10. Regular Expressions: `re`

The `re` module is essential for data normalization, extraction, and validation. In data engineering you encounter raw vendor text, log lines, and dirty CSV fields that need structured extraction.

**Key functions:**

| Function | What it does |
| --- | --- |
| `re.match(pattern, s)` | Match at the start of the string only |
| `re.search(pattern, s)` | Find first match anywhere in string |
| `re.findall(pattern, s)` | Return list of all non-overlapping matches |
| `re.finditer(pattern, s)` | Iterator of match objects (preferred for large input) |
| `re.sub(pattern, repl, s)` | Replace all matches |
| `re.split(pattern, s)` | Split on pattern |
| `re.compile(pattern)` | Compile for repeated use — do this in production |

**Pattern syntax to know cold:**

| Pattern | Matches |
| --- | --- |
| `.` | Any char except newline |
| `\d` / `\D` | Digit / non-digit |
| `\w` / `\W` | Word char (alphanum + `_`) / non-word |
| `\s` / `\S` | Whitespace / non-whitespace |
| `^` / `$` | Start / end of string |
| `+` / `*` / `?` | One-or-more / zero-or-more / optional |
| `{n,m}` | Between n and m repetitions |
| `(group)` | Capture group — indexed from 1 |
| `(?P<name>...)` | Named capture group — use `.groupdict()` |
| `(?:...)` | Non-capturing group |
| `[abc]` / `[^abc]` | Character class / negated class |
| `a\|b` | Alternation (a or b) |

**Industry uses:**
- Parse structured fields from unstructured log lines
- Validate symbol, file naming, and field format conventions
- Normalize separators (`/`, `-`, `_`, `.`) in vendor data
- Extract prices, quantities, and IDs from text blobs


In [ ]:
import re

# Compile patterns once for repeated use — better performance in hot paths
SYMBOL_PATTERN = re.compile(r"^[A-Z]{6}$")
PRICE_PATTERN = re.compile(r"\d+\.\d+")
LOG_PATTERN = re.compile(
    r"(?P<ts>\d{4}-\d{2}-\d{2}T[\d:+]+)\s+"
    r"(?P<level>\w+)\s+"
    r"(?P<symbol>[A-Z]{6})\s+"
    r"price=(?P<price>[\d.]+)"
)

# Validation with match (anchored to start only)
symbols_to_check = ["EURUSD", "EUR_USD", "eurusd", "EURUS1", "GBPUSD"]
valid_symbols = [s for s in symbols_to_check if SYMBOL_PATTERN.match(s)]

# Extraction using named groups
raw_log = "2026-04-04T08:15:00+00:00 INFO EURUSD price=1.0845 volume=1000"
match = LOG_PATTERN.search(raw_log)
extracted = match.groupdict() if match else {}

# findall: extract all prices from a multi-symbol blob
price_blob = "EURUSD: 1.0845/1.0847  USDJPY: 145.12/145.14  GBPUSD: 1.2641/1.2644"
all_prices = PRICE_PATTERN.findall(price_blob)

# sub: normalize separator characters in symbol strings
dirty_symbols = ["EUR/USD", "USD_JPY", "GBP-USD", "AUD.USD"]
clean_symbols = [re.sub(r"[/_\-.]", "", s).upper() for s in dirty_symbols]

# split on multiple delimiters (pipes, semicolons, spaces)
raw_row = "EURUSD|1.0845;BUY 1000"
parts = re.split(r"[|; ]+", raw_row)

# Validate file naming convention
FILE_PATTERN = re.compile(r"^prices_\d{4}-\d{2}-\d{2}\.csv$")
file_names = ["prices_2026-04-04.csv", "prices_bad.csv", "data_2026.csv"]
valid_files = [f for f in file_names if FILE_PATTERN.match(f)]

{
    "valid_symbols": valid_symbols,
    "extracted_from_log": extracted,
    "all_prices_found": all_prices,
    "clean_symbols": clean_symbols,
    "split_parts": parts,
    "valid_files": valid_files,
}


## 11. Dates, Times, and Timezones: `datetime` and `zoneinfo`

Datetime handling is one of the most error-prone areas in data engineering. Silent bugs here corrupt aggregations and time-series analysis.

**Core types:**
- `datetime.datetime` — specific point in time (date + time)
- `datetime.date` — calendar date only
- `datetime.timedelta` — a duration
- `datetime.timezone` — fixed UTC offset
- `zoneinfo.ZoneInfo` — IANA named timezone (Python 3.9+, the right way for DST-aware zones)

**Rules to internalize:**
- Always be explicit about whether a datetime is timezone-aware or naive
- Never mix aware and naive datetimes in arithmetic or comparisons
- Store and transmit in UTC; convert to local only at the display boundary
- Use `datetime.fromisoformat()` for ISO 8601 strings
- Use `strftime` / `strptime` for custom format strings

**Common failure modes in data engineering:**
- Naive datetimes silently treated as local time by database drivers
- DST transitions creating gaps or duplicates in time-series windows
- Joining datasets that use different timezone conventions without normalizing to UTC first

**Interview questions:**
- What is the difference between a timezone-aware and a timezone-naive datetime?
- Why should you normalize to UTC before storing event timestamps?
- What is the difference between `replace(tzinfo=...)` and `astimezone(...)`?
- How would you floor a timestamp to the nearest minute for time-bucketing?


In [ ]:
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo

UTC = timezone.utc
LONDON = ZoneInfo("Europe/London")
NEW_YORK = ZoneInfo("America/New_York")
TOKYO = ZoneInfo("Asia/Tokyo")

# Parse ISO 8601 strings — the dominant format in APIs and logs
ts_utc = datetime.fromisoformat("2026-04-04T08:15:00+00:00")
ts_naive = datetime.fromisoformat("2026-04-04T08:15:00")  # no tzinfo — dangerous

# Make a naive datetime explicit
ts_assumed_utc = ts_naive.replace(tzinfo=UTC)

# Convert between timezones
ts_london = ts_utc.astimezone(LONDON)
ts_new_york = ts_utc.astimezone(NEW_YORK)
ts_tokyo = ts_utc.astimezone(TOKYO)

# Datetime arithmetic
trade_ts = datetime(2026, 4, 4, 8, 15, tzinfo=UTC)
settlement_ts = trade_ts + timedelta(days=2)
days_until_expiry = (date(2026, 6, 30) - date(2026, 4, 4)).days

# Formatting — know these patterns cold
formatted = {
    "iso": ts_utc.isoformat(),
    "human": ts_utc.strftime("%Y-%m-%d %H:%M:%S %Z"),
    "date_only": ts_utc.strftime("%Y-%m-%d"),
    "filename_safe": ts_utc.strftime("%Y%m%d_%H%M%S"),
}

# Floor to minute — for time-bucketing in VWAP / OHLC
def floor_to_minute(ts: datetime) -> datetime:
    return ts.replace(second=0, microsecond=0)

# Session boundary check
def is_london_hours(ts: datetime) -> bool:
    local = ts.astimezone(LONDON)
    return local.hour in range(8, 17)

# strptime: parse custom format strings (not ISO)
custom_ts = datetime.strptime("04-Apr-2026 08:15:00", "%d-%b-%Y %H:%M:%S").replace(tzinfo=UTC)

{
    "ts_london": ts_london.isoformat(),
    "ts_new_york": ts_new_york.isoformat(),
    "ts_tokyo": ts_tokyo.isoformat(),
    "settlement": settlement_ts.isoformat(),
    "days_until_expiry": days_until_expiry,
    "formatted": formatted,
    "is_london_hours": is_london_hours(trade_ts),
    "floored": floor_to_minute(trade_ts).isoformat(),
    "custom_parsed": custom_ts.isoformat(),
}


## 12. Mini Lab

Solve this using only built-ins and the standard library shown above.

Input: a small batch of raw trade rows.

Tasks:
1. Normalize `symbol`, `venue`, and `event_type`.
2. Parse prices and timestamps (timezone-aware, UTC).
3. Deduplicate on `(symbol, venue, ts)`.
4. Build `symbol -> latest_price`.
5. Count events by `event_type`.
6. Use a generator expression for total notional (memory-efficient).
7. Emit both a CSV preview and a JSON summary report.
8. Wrap the file write in a context manager.

Interview framing:
- Why did you choose `set` for dedupe?
- Why did you choose a generator expression instead of a list for total notional?
- Where would a decorator be useful in this pipeline?
- Which regex pattern would you use to validate the symbol field?
- How would you handle timestamps from two vendors using different timezones?

## 13. Exit Checklist

Do not move on until you can answer these without searching:

**Containers:**
- When is `list` better than `set`, and when is `set` better than `list`?
- Why is `deque` better than `list` for queue workloads?
- Why is a tuple acceptable as a dictionary key but a list is not?
- What are the most important methods on `str`, `list`, `set`, and `dict`?

**Strings and formatting:**
- What is the difference between `find` and `index`?
- When would you use `partition` instead of `split`?
- How do f-strings, `str.format`, and `format()` relate to each other?
- How would you format money, percentages, aligned columns, and timestamps?

**Comprehensions:**
- What are the four types of comprehensions?
- What is the difference between a list comprehension and a generator expression?
- When should you use a generator expression instead of a list comprehension?

**Decorators:**
- What does `@functools.wraps` do and why does it matter?
- What is the difference between a decorator and a decorator factory?
- When would you reach for `functools.lru_cache`?
- What does `functools.partial` return?

**Context managers:**
- What does `__exit__` receive when an exception is raised inside the `with` block?
- What is the difference between `contextlib.contextmanager` and a class-based context manager?
- What does `contextlib.suppress` do?

**Regex:**
- What is the difference between `re.match` and `re.search`?
- What is a named capture group and how do you access it?
- Why should you `re.compile` patterns that are used repeatedly?

**Datetime:**
- What is the difference between a timezone-aware and a naive datetime?
- Why should you store timestamps in UTC?
- What is the difference between `.replace(tzinfo=...)` and `.astimezone(...)`?
- How would you floor a datetime to the nearest minute?

**General:**
- What is a shallow copy of a list or dictionary?
- Can you normalize, deduplicate, and aggregate a small dataset using only the standard library?

## Official References Used To Build This Notebook

- Python Tutorial: Data Structures
  https://docs.python.org/3/tutorial/datastructures.html
- Python Standard Library: Built-in Types
  https://docs.python.org/3/library/stdtypes.html
- Python Standard Library: `collections`
  https://docs.python.org/3/library/collections.html
- Python Standard Library: `pathlib`
  https://docs.python.org/3/library/pathlib.html
- Python Standard Library: `csv` and `json`
  https://docs.python.org/3/library/csv.html
  https://docs.python.org/3/library/json.html
- Python Standard Library: `itertools`
  https://docs.python.org/3/library/itertools.html
- Python Standard Library: Format String Syntax
  https://docs.python.org/3/library/string.html#format-string-syntax
- Python Standard Library: `functools`
  https://docs.python.org/3/library/functools.html
- Python Standard Library: `contextlib`
  https://docs.python.org/3/library/contextlib.html
- Python Standard Library: `re`
  https://docs.python.org/3/library/re.html
- Python Standard Library: `datetime`
  https://docs.python.org/3/library/datetime.html
- Python Standard Library: `zoneinfo`
  https://docs.python.org/3/library/zoneinfo.html
- Python HOWTO: Regular Expression HOWTO
  https://docs.python.org/3/howto/regex.html